# 33 · Cost and latency: caps, caching, routing and batch
Level L6 · Part 6 · core · ~15 min · builds on 32

## Goal
Price one support ticket from the usage in its reply, see how a cached prefix lowers the bill (and what a cache miss
costs), send the easy cases away from the model, and plan a latency budget. The budget is a plan to compare with real
timings from lesson 29. Three model calls in all, each recorded once.

## Why it matters
Lesson 32 made the refund safe to retry. A safe agent can still be expensive. Run the cell below. It is arithmetic
only, with no model call. A bug makes the agent repeat its model call 40 times per ticket, and nothing caps the loop.
The prices are the model's standard prices, per 1M tokens.

In [1]:
PRICE_IN, PRICE_OUT = 0.25, 1.50          # USD per 1M tokens (ai.google.dev pricing, checked 2026-10-20)
PRICE_CACHED = 0.025                      # USD per 1M cached input tokens: 10% of the input price (Google: 90% off)
calls_per_ticket = 40                     # the loop never stops and repeats its model call
prompt_tokens, reply_tokens = 4_000, 200  # the whole policy text is sent again on every call
tickets = 20_000                          # one month of support tickets, the same scale as Step 4
per_call = (prompt_tokens * PRICE_IN + reply_tokens * PRICE_OUT) / 1_000_000
print(f"one ticket: ${per_call * calls_per_ticket:.2f}")
print(f"{tickets:,} tickets in a month: ${per_call * calls_per_ticket * tickets:,.2f}, and nobody noticed until the invoice")

one ticket: $0.05
20,000 tickets in a month: $1,040.00, and nobody noticed until the invoice


**What it makes easy:** a cap stops the loop, a cached prefix makes the repeated text cheap, routing sends the simple
cases away from the model, and batch runs the offline work at half price. **When you don't need it:** a demo with ten
tickets and no customers.

## The idea
A model bill is tokens times price, and the price is per token, for input and for output. Output costs six times as
much as input on this model (1.50 against 0.25), so a long reply is the costly part. Four levers follow from that:
**caps** stop a run before it overspends, **caching** pays less for text the service has seen before, **routing**
keeps the model away from work it does not need to do, and **batch** trades time for half the price.

Analogy: a taxi meter. You can see the fare grow on every ride, you can set a limit before you get in, and you choose
the cheaper route when there is one.

## Picture
```mermaid
flowchart LR
  T[Ticket] --> R{Router label}
  R -->|status| L[Order lookup: no model]
  R -->|refund| F[Refund path: lesson 14 approval]
  R -->|other| M[Model answer: capped reply, cached policy]
  M --> S[Spend check and usage log]
```

## Step by step
**Step 1 · set up the client and price one ticket (needs-llm).** Every reply already carries its usage, so the bill is
arithmetic on numbers you have. `cached_content_token_count` is a field of the reply's `usage_metadata`: the number of
prompt tokens the service served from its cache instead of reading them again. The model cell below is the only call
in this step. The key is read from `.env`, and later runs replay the recording.

The `llm_replay` and `tut_env` lines are course helpers. In your own project, replace them with
`os.environ["GOOGLE_CLOUD_API_KEY"]` and a plain `genai.Client`: that is what you would do there.

In [2]:
import sys; sys.path.insert(0, "..")
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("33_cost-and-latency")

from google import genai  # the official Google SDK
from google.genai import types
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"  # the course's default model, as in lessons 02 to 32
client = genai.Client(
    vertexai=True,
    api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(
        attempts=3, initial_delay=5.0, max_delay=30.0)),  # the SDK's own retry for 429 and 5xx, 3 tries as in lesson 32
)

LLM mode: replay (33_cost-and-latency.yaml)


The shared policy is a real document in the course data: the FTC's business guide to the mail, internet or telephone
order rule (about 5,000 words, public US government text; see `data/SOURCES.md`). Brightside follows it for shipping.
It goes first in every request, so the service can reuse it. The question goes last. The reply is capped with
`max_output_tokens`, a per-request cap.

In [3]:
from pathlib import Path
from google.genai import types  # the SDK's config types: no model call here

FTC_RULE = Path("../data/ftc_30_day_rule.txt").read_text(encoding="utf-8")
print(FTC_RULE[:160].replace("\n", " "), "...", len(FTC_RULE.split()), "words")
INSTRUCTIONS = """You are the support desk assistant for Brightside Outfitters, an online shop for outdoor clothing.
Answer in at most three short sentences. Never promise a refund yourself: say that a refund needs approval.
For shipping and cancellation questions, use the rule text below. If it does not cover the question, say so.

"""
POLICY = INSTRUCTIONS + FTC_RULE


def ask(question: str, max_out: int = 400):
    """One model call with the shared policy first, the question last, and a cap on the reply length."""
    return client.models.generate_content(
        model=MODEL, contents=question,
        config=types.GenerateContentConfig(system_instruction=POLICY, max_output_tokens=max_out, temperature=0))

How to Comply With the Rule  The following information will help you comply with the Rule.  What You Should Know Before You Make a Shipment Representation  When ... 5029 words


**The first ticket (needs-llm).** This is the first model call. The cells that price its reply are tagged the same way,
so with the model skipped they are skipped too and nothing fails.

In [4]:
ticket_1 =ask("Order 2207 arrived with a torn box. What do I tell the customer?")  # the first model call
print("reply:", ticket_1.text)

reply: I am sorry to hear that your order arrived in a damaged box. Please contact our support team with photos of the damage so we can review your case. Any potential refund would need to be submitted for approval.


**The bill from the usage.** `cost_of` prices the cached part at the cached price and the rest at the input price.
`report` prints the counts next to the cost, so you can check them against the reply.

In [5]:
def cost_of(resp) -> float:
    """The bill for one reply, from its usage: cached prompt tokens at the cached price, the rest at the input price."""
    u = resp.usage_metadata
    cached = u.cached_content_token_count or 0       # prompt tokens served from the cache
    fresh = u.prompt_token_count - cached            # the rest, at the full input price
    out = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)  # thinking is billed as output
    return (fresh * PRICE_IN + cached * PRICE_CACHED + out * PRICE_OUT) / 1_000_000


def report(name: str, resp) -> float:
    u = resp.usage_metadata
    print(f"{name}: prompt {u.prompt_token_count} tokens (cached {u.cached_content_token_count or 0}), "
          f"reply {u.candidates_token_count or 0} + thinking {u.thoughts_token_count or 0}, cost ${cost_of(resp):.6f}")
    return cost_of(resp)

**Price the first reply (needs-llm).** The line below prices ticket 2207 from its recorded usage.

In [6]:
_ = report("ticket 2207", ticket_1)  # the value is returned, not printed: _ keeps the cell quiet

ticket 2207: prompt 6289 tokens (cached 6109), reply 42 + thinking 0, cost $0.000261


**Step 2 · a second ticket with the same policy (needs-llm).** The question differs, but the policy does not, and this
ticket misses the cache. In the recorded run, ticket 2207 hit it (6,109 of 6,289 prompt tokens cached), because an
earlier request had already sent the same policy text. Ticket 1093 had none cached, so it cost about six times more for
almost the same work. Caching is implicit: the service decides, and this lesson does not switch it on. To raise your
hit rate, keep the prefix identical and first in the request, and send related requests close together. Explicit
caching is out of scope. Read the cached count on every reply.

In [7]:
ticket_2 = ask("Order 1093 never came. What do I tell the customer?")  # the second model call
_ = report("ticket 1093", ticket_2)  # a cache miss: cached 0, so the same work costs more

ticket 1093: prompt 6286 tokens (cached 0), reply 56 + thinking 0, cost $0.001656


**Step 3 · route before you call (needs-llm).** A short call labels the message with one word. Code then sends it down
a branch. An order status question needs no model at all: the lookup answers it. An unknown label goes to a safe
default, the model answer. Routing here saves the long prompt, not model size: the router uses the same model, but
with no policy and a 50-token reply cap. This is the last model call of the notebook, so the recording stops here.

In [8]:
ROUTES = {"status": "order lookup (no model)", "refund": "refund path (lesson 14 approval)",
          "other": "model answer (capped reply, cached policy)"}
label_reply = client.models.generate_content(  # the third model call: short prompt, no policy, tiny reply
    model=MODEL, contents="Label this support message with one word: status, refund or other.\n\nMessage: Where is my parcel for order 1093?",
    config=types.GenerateContentConfig(max_output_tokens=50, temperature=0))
llm_replay.stop()  # the notebook makes no further model calls
word = "".join(ch for ch in (label_reply.text or "").lower() if ch.isalpha())
branch = word if word in ROUTES else "other"  # an unknown label goes to the safe default
print("label:", word or "(empty)", "-> route:", ROUTES[branch])

label: status -> route: order lookup (no model)


The order status comes from a lookup, not a model. A tool-result cache stores the answer, so a repeat question costs
nothing. The fictional orders are the same as in lesson 06.

In [9]:
from functools import lru_cache

ORDER_STATUS = {"1093": "in transit", "2207": "delivered"}  # fictional rows, the same as lesson 06
LOOKUPS = {"n": 0}


@lru_cache(maxsize=256)  # the tool-result cache: the same order number is looked up once
def order_status(order_id: str) -> str:
    LOOKUPS["n"] += 1
    return ORDER_STATUS.get(order_id, "unknown order")


print(order_status("1093"), "|", order_status("1093"), "| lookups made:", LOOKUPS["n"])

in transit | in transit | lookups made: 1


**Step 4 · the monthly bill (no call, worked example).** 20,000 tickets a month, each with about 6,300 prompt tokens
(6,250 of them the shared policy) and a 50-token reply. The cached line is a best case: it assumes every ticket finds
the policy cached, as ticket 2207 did in Step 1. Step 2 shows a miss, so the real bill sits between the two lines.
Batch is half price. The pricing page does not say whether batch and cache discounts stack.

In [10]:
TICKETS = 20_000
FRESH, CACHED, REPLY = 50, 6_250, 50    # tokens per ticket: the question, the shared policy, the reply


def month(price_cached: float | None, batch: float = 1.0) -> float:  # batch 0.5 = half price
    """Monthly USD. price_cached=None means no cache: the policy is paid at the full input price."""
    prompt = FRESH * PRICE_IN + (CACHED * PRICE_IN if price_cached is None else CACHED * price_cached)
    return TICKETS * (prompt + REPLY * PRICE_OUT) / 1_000_000 * batch


print(f"{'no levers':<34}${month(None):>9.2f}")
print(f"{'best case: policy always cached':<34}${month(PRICE_CACHED):>9.2f}")
print(f"{'batch only, offline review':<34}${month(None, 0.5):>9.2f}")

no levers                         $    33.00
best case: policy always cached   $     4.88
batch only, offline review        $    16.50


**Step 5 · a latency budget, and tools that run at the same time (no model).** A budget is a plan: each stage gets a
share of the time the user will wait. Here, three lookups each take 0.2 s, simulated with `sleep` because a tool is not
a model. Run one after another they take 0.6 s. Run together, they take about the time of the slowest one. Lesson 29
measures real timings to compare with this plan.

In [11]:
import time
from concurrent.futures import ThreadPoolExecutor


def lookup(name: str) -> str:
    time.sleep(0.2)  # a simulated tool call: 0.2 s each
    return name


BUDGET_MS = {"router label": 300, "tool lookups": 200, "model answer": 1200, "format": 50}  # planned, not measured
print("budget total:", sum(BUDGET_MS.values()), "ms of 2000 ms")

names = ["order", "refund rules", "carrier status"]
t0 = time.perf_counter()
[lookup(n) for n in names]
one_by_one = time.perf_counter() - t0
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as pool:
    list(pool.map(lookup, names))
together = time.perf_counter() - t0
print(f"one after another: {one_by_one:.1f} s · together: {together:.1f} s")

budget total: 1750 ms of 2000 ms


one after another: 0.6 s · together: 0.2 s


## Under the hood
No library in this stack enforces a per-run spend cap, so the guard is written by hand. Gateways such as LiteLLM add the
same check per team or per key (lesson 34). The guard is checked before each call, so the call that would cross the cap
is never sent. The spend comes from the usage in each reply, not from a guess.

In [12]:
class BudgetExceeded(Exception): pass  # raised before the call that would cross the cap


class SpendGuard:
    """A per-run spend cap: check before each model call, add each reply's cost after it."""

    def __init__(self, max_usd: float):
        self.max_usd, self.spent = max_usd, 0.0

    def check(self) -> None:
        if self.spent >= self.max_usd:
            raise BudgetExceeded(f"spent ${self.spent:.6f} of the ${self.max_usd:.6f} cap; no call made")

    def add(self, resp) -> None:
        self.spent += cost_of(resp)

**The demo (needs-llm).** It adds the first reply's real cost to a small cap, then shows the next check refusing.

In [13]:
guard =SpendGuard(max_usd=0.0002)  # a small cap for the demo: the first reply costs more than this
guard.check()
guard.add(ticket_1)                 # the first reply's real cost
try:
    guard.check()                   # the next call would start past the cap
except BudgetExceeded as err:
    print("stopped:", err)

stopped: spent $0.000261 of the $0.000200 cap; no call made


The demo above only shows the refusal: the first reply's cost is added, and the next check stops the run before any
call is made. `guarded_ask` puts the same three steps around a real model call. It is defined here but not called, so
the notebook makes no new model call. In your own code, every model call goes through a function like this.

In [14]:
def guarded_ask(guard: SpendGuard, question: str):
    guard.check()           # 1. refuse before the call if the cap is already spent
    resp = ask(question)    # 2. the model call, the same as in Step 1
    guard.add(resp)         # 3. add what this reply really cost
    return resp


print("guarded_ask ready; it makes a model call only when you call it")

guarded_ask ready; it makes a model call only when you call it


## At real scale
- **Output costs the most.** Here output is six times the input price. Cap the reply, not only the prompt.
- **Cache the stable text, first.** A cached token is billed at a tenth. The cache is best effort, so read
  `cached_content_token_count` on every reply and alert when it drops. → lesson 15 (context).
- **Batch is for offline work.** The Gemini Batch API takes half the price but runs later. Use it for nightly reviews
  and evals (→ lesson 28).
- **Caps belong at the gateway too.** A code cap stops one app. A gateway holds a budget per team or key and a fallback
  model (→ lesson 34).
- **Latency is a budget.** Streaming shows the first words early. Parallel tool calls wait for the slowest tool, not
  the sum. Measure the tail (→ lesson 29).

## Decision guide
| Situation | Approach | Why |
|---|---|---|
| A loop that could run forever | step cap and spend cap, checked before each call | the call that crosses the cap is never sent |
| The same long policy on every call | put it first, keep it identical, read the cached count | a cached token costs a tenth of an input token, when the service hits the cache |
| A question that a lookup answers | no model: a tool, cached by key | no tokens, no latency for the model |
| A simple label before a big task | a short prompt labels, code routes | the label costs a fraction of the answer's tokens |
| Offline review of many tickets | batch at half price | nobody waits for the answer |
| Users waiting on a reply | stream, run tools in parallel, plan the budget per stage | the first words arrive early |

## Common mistakes
1. **A cap that is not a number.** The config checks the type when it is built. A cap of `"four hundred"` is refused
   before any call is sent, and the error names the field.

In [15]:
types.GenerateContentConfig(max_output_tokens="four hundred")  # the cap must be a number

ValidationError: 1 validation error for GenerateContentConfig
max_output_tokens
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='four hundred', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_parsing

The fix is `max_output_tokens=400`. No request was made, so no money moved.

2. **Pricing a cached prompt at the full input price.** That overstates the bill and hides the saving the cache gives.
   Read `cached_content_token_count` and price it apart, as `cost_of` does.
3. **Sending every message to the biggest model.** Most messages are a status question or a label. Route them first, so the
   big model answers only the cases that need it.

## Try it
A call has 2,000 prompt tokens, of which 1,500 are cached, and a 200-token reply. Predict its cost in USD with the prices
in this notebook. Then check your answer with the code. Write `cost` in the code cell.

In [16]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [17]:
# 500 fresh tokens at 0.25, 1,500 cached at 0.025, 200 reply tokens at 1.50, all per 1M tokens
cost = (500 * PRICE_IN + 1_500 * PRICE_CACHED + 200 * PRICE_OUT) / 1_000_000  # predicted: 0.0004625
print(f"cost: ${cost:.7f}")

cost: $0.0004625


## Recap
- A bill is tokens times price. Output costs six times the input, so cap the reply too.
- A spend cap is checked before each call, so the call that would cross it is never sent.
- A cached token costs a tenth of the input price, but only on a hit. Keep the shared text identical and first.
- Route simple work away from the model, cache tool results, and use batch for offline jobs.
- A latency budget is a plan per stage. Parallel tools cost the slowest one, not the sum.

**Interview:** "A support agent's bill doubled overnight. What do you check first?"
Strong answer: "Group the cost by run and by ticket from the usage in each reply. Look for a loop or a prompt that
stopped being cached. Then add a step cap and a spend cap checked before each call, and a budget per team at the gateway."